In [140]:
import numpy as np
import pandas as pd

In [141]:
corpus = """i like cats
i love cats
i love dogs
i hate trees
i need help
he does not love you
big hairy kitten
i only love cars"""

In [142]:

docs = corpus.strip('').split('\n')
docsAct = []

for doc in docs:
    docsAct.append(doc.split(' '))


In [143]:
# tf
for doc in docs :
    for word in doc:
        pass
print(set(docsAct[1]))


{'cats', 'love', 'i'}


In [144]:
vocab = set()
for doc in docsAct:
    vocab.update(doc)
print(vocab)
mat = np.array([
    sorted(list(vocab))
], dtype=object)

{'does', 'hate', 'not', 'hairy', 'trees', 'cats', 'help', 'need', 'dogs', 'i', 'he', 'kitten', 'only', 'you', 'love', 'big', 'like', 'cars'}


In [145]:
print(mat)

[['big' 'cars' 'cats' 'does' 'dogs' 'hairy' 'hate' 'he' 'help' 'i'
  'kitten' 'like' 'love' 'need' 'not' 'only' 'trees' 'you']]


In [146]:
word_to_col = {}
i = 0

for word in mat[0]:
    word_to_col[word] = i
    i+=1

for doc in docsAct:
    row = np.zeros(len(mat[0]), dtype=int)
    for word in doc:
        tf = doc.count(word)
        col = word_to_col[word]
        row[col] = tf
    mat = np.vstack((mat, row))

In [147]:
print(mat)

[['big' 'cars' 'cats' 'does' 'dogs' 'hairy' 'hate' 'he' 'help' 'i'
  'kitten' 'like' 'love' 'need' 'not' 'only' 'trees' 'you']
 [0 0 1 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 1 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0]
 [0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 0 0 1]
 [1 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 1 0 0 1 0 0 1 0 0]]


In [148]:
#idf 
N = len(docs)
# we need to get df(number if docs that have the term in them)
df_list = []
for word in sorted(vocab):
    dfWord = 0
    for doc in docsAct:
        if word in doc:
            dfWord+=1
    df_list.append(dfWord)
print(df_list)

[1, 1, 2, 1, 1, 1, 1, 1, 1, 6, 1, 1, 4, 1, 1, 1, 1, 1]


In [149]:
print(len(df_list))
for index,row in enumerate(mat[0]):
    for i in range(1,N+1):
        if mat[i][index] != 0:
            idf = np.log(N/df_list[index])
            tf_idf = idf* mat[i][index]
            mat[i][index] = tf_idf


18


In [150]:
print(mat)

[['big' 'cars' 'cats' 'does' 'dogs' 'hairy' 'hate' 'he' 'help' 'i'
  'kitten' 'like' 'love' 'need' 'not' 'only' 'trees' 'you']
 [0 0 1.3862943611198906 0 0 0 0 0 0 0.28768207245178085 0
  2.0794415416798357 0 0 0 0 0 0]
 [0 0 1.3862943611198906 0 0 0 0 0 0 0.28768207245178085 0 0
  0.6931471805599453 0 0 0 0 0]
 [0 0 0 0 2.0794415416798357 0 0 0 0 0.28768207245178085 0 0
  0.6931471805599453 0 0 0 0 0]
 [0 0 0 0 0 0 2.0794415416798357 0 0 0.28768207245178085 0 0 0 0 0 0
  2.0794415416798357 0]
 [0 0 0 0 0 0 0 0 2.0794415416798357 0.28768207245178085 0 0 0
  2.0794415416798357 0 0 0 0]
 [0 0 0 2.0794415416798357 0 0 0 2.0794415416798357 0 0 0 0
  0.6931471805599453 0 2.0794415416798357 0 0 2.0794415416798357]
 [2.0794415416798357 0 0 0 0 2.0794415416798357 0 0 0 0
  2.0794415416798357 0 0 0 0 0 0 0]
 [0 2.0794415416798357 0 0 0 0 0 0 0 0.28768207245178085 0 0
  0.6931471805599453 0 0 2.0794415416798357 0 0]]


In [151]:
def cosineSim(vec1 , vec2):
    dotPro = np.dot(vec1 , vec2)

    norm1 = np.sqrt(np.sum(vec1**2))
    norm2 = np.sqrt(np.sum(vec2**2))

    return dotPro / (norm1*norm2)

In [163]:
def getSims(indexDoc):
    doc = docsAct[indexDoc]
    sims = []
    for index,row in enumerate(mat):
        if index==0:
            continue
        if index != indexDoc:
            sim = cosineSim(mat[indexDoc], row)
            sims.append((sim, docs[index - 1]))
    return sims
        

In [164]:
print(f"chosen doc is:: ({str(docs[indexDoc-1])})")
for sim , doc in sorted(sims , reverse = True):
    print(sim , "->",doc)

chosen doc is:: (i like cats)
0.5054763796574575 -> i love cats
0.014881131301506005 -> i love dogs
0.011133716724411146 -> i need help
0.011133716724411146 -> i hate trees
0.010839468331747966 -> i only love cars
0.0 -> he does not love you
0.0 -> big hairy kitten


In [ ]:
corpus = """i like cats 
i love cats
i love dogs
i hate trees
i need help
he does not love you
big hairy kitten
i only love cars
the small kitten likes milk
cats and dogs can live together
my dog loves playing outside
the red car is very fast
i need a new car
to be or not to be that's the f question lil bro
she likes fast cars
the big dog chased the small cat
i love my little kitten
help me find my lost dog
the forest has many tall trees
to pee or not to pee
i enjoy driving fast cars
cats are cute and fluffy
dogs are loyal animals
the black kitten is sleeping
i need help with my car
she hates noisy dogs
the little cat climbed the tree"""

In [181]:
def build_tf_idf(corpus):
    docs = corpus.strip().split('\n')
    docsAct = []

    for doc in docs:
        docsAct.append(doc.split(' '))

    vocab = set()
    for doc in docsAct:
        vocab.update(doc)

    mat = np.array([
        sorted(list(vocab))
    ], dtype=object)

    word_to_col = {}
    i = 0
    for word in mat[0]:
        word_to_col[word] = i
        i += 1

    for doc in docsAct:
        row = np.zeros(len(mat[0]), dtype=int)

        for word in doc:
            tf = doc.count(word)
            col = word_to_col[word]
            row[col] = tf

        mat = np.vstack((mat, row))

    N = len(docs)

    df_list = []
    for word in sorted(vocab):
        dfWord = 0

        for doc in docsAct:
            if word in doc:
                dfWord += 1

        df_list.append(dfWord)

    for index, row in enumerate(mat[0]):
        for i in range(1, N + 1):
            if mat[i][index] != 0:
                idf = np.log(N / df_list[index])
                tf_idf = idf * mat[i][index]
                mat[i][index] = tf_idf

    return mat, docs

In [182]:
def search(mat, docs, indexDoc):
    sims = []
    query_row = indexDoc + 1 
    for index, row in enumerate(mat):
        if index == 0:
            continue

        if index != query_row:
            sim = cosineSim(mat[query_row], row)
            sims.append((sim, docs[index - 1]))

    sims.sort(reverse=True)

    return sims

In [184]:
mat, docs = build_tf_idf(corpus)
chosen = 13
results = search(mat, docs, chosen)

print(f"og doc is ({docs[chosen]})")
for sim, doc in results:

    print(f"{sim:.3f} -> {doc}")

og doc is (to be or not to be that's the f question lil bro)
0.363 -> to pee or not to pee
0.065 -> he does not love you
0.048 -> the little cat climbed the tree
0.047 -> the big dog chased the small cat
0.028 -> the small kitten likes milk
0.027 -> the black kitten is sleeping
0.025 -> the red car is very fast
0.021 -> the forest has many tall trees
0.000 -> she likes fast cars
0.000 -> she hates noisy dogs
0.000 -> my dog loves playing outside
0.000 -> i only love cars
0.000 -> i need help with my car
0.000 -> i need help
0.000 -> i need a new car
0.000 -> i love my little kitten
0.000 -> i love dogs
0.000 -> i love cats
0.000 -> i like cats
0.000 -> i hate trees
0.000 -> i enjoy driving fast cars
0.000 -> help me find my lost dog
0.000 -> dogs are loyal animals
0.000 -> cats are cute and fluffy
0.000 -> cats and dogs can live together
0.000 -> big hairy kitten
